
# **2.5 Automatic Differentiation**
Deep learning frameworks accelerate the process of finding derivatives by automatically computing them—a process known as automatic differentiation. **In practice, based on a well-designed model, the system constructs a computational graph to track which data points are combined through which operations to produce the output.** Automatic differentiation enables the system to subsequently backpropagate the gradients. Here, backpropagation refers to tracing the entire computational
graph to fill in the partial derivatives for each parameter.

## **2.5.1 A simple example**
As an illustrative example, suppose we want to differentiate the function $y = 2x^⊤x$ with respect to the column vector **x**. First, we create the variable x and assign it an initial value.

In [1]:
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

Before we compute the gradient of y with respect to x, we need a place to store the gradient. **It is important that we do not allocate new memory every time we take the derivative of a parameter.** Since we often update the same parameter thousands or even tens of thousands of times, allocating new memory each time could quickly exhaust our memory. **Note that the gradient of a scalar function with respect to a vector x is a vector and has the same shape as x.**

In [2]:
x.requires_grad_(True) # equivalent to x = torch.arange(4.0, requires_grad=True)
x.grad # the default value is None

Now calculate y.

In [3]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

x is a vector of length 4. We calculate the dot product of x and x, which yields the scalar output we assign to y. Next, we call the backpropagation function to automatically compute the gradients of y with respect to each component of x, and print these gradients.

In [4]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

The gradient of the function y = 2**x**$^T$**x** with respect to **x** should be **4x**.

In [5]:
x.grad == 4 * x

tensor([True, True, True, True])

Now let's calculate another function of x.

In [6]:
# By default, PyTorch accumulates gradients, so we need to clear the previous values
x.grad.zero_()
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

## **2.5.2 Backpropagation for Non-Scalar Variables**
When y is not a scalar, the most natural interpretation of the derivative of the vector y with respect to the vector x is a matrix. For higher-order and higher-dimensional y and x, the result of differentiation can be a higher-order tensor.<br><br>
However, while these more exotic objects do indeed appear in advanced machine learning (including deep learning), when we refer to the backward computation of a vector, we are typically trying to compute the derivatives of the loss function for each component in a batch of training samples. **Here, our goal is not to compute a differential matrix, but rather to compute the sum of the partial derivatives for each sample in the batch individually.**

In [7]:
# When calling backward on a non-scalar, you must pass a gradient parameter, which specifies the gradient of the differential function with respect to self.
# In this example, we only want to compute the sum of the partial derivatives, so passing a gradient of 1 is appropriate.
x.grad.zero_()
y = x * x
y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

## **2.5.3 Separate Calculations**
**Sometimes, we want to move certain computations outside the computational graph of a record.** For example, suppose y is computed as a function of x, and z is computed as a function of both y and x. Imagine we want to compute the gradient of z with respect to x, but for some reason, we want to treat y as a constant and only consider the role x plays after y has been computed.<br><br>
**Here, we can isolate y to obtain a new variable u, which has the same value as y but discards any information about how y is computed in the computational graph. In other words, the gradient does not flow backward through u to x.** Therefore, the backpropagation function below computes the partial derivative of z = u * x with respect to x, while treating u as a constant, rather than the partial derivative of z = x * x * x with respect to x.

In [8]:
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

Since we have recorded the result of the calculation for y, we can then apply backpropagation to y to obtain the derivative of y = x$*$x with respect to x, which is 2$*$x.

In [9]:
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

## **2.5.4 Gradient Calculation in Python Control Flow**
**One advantage of using automatic differentiation is that even if the computational graph of a construct function involves Python control flow (such as conditions, loops, or arbitrary function calls), we can still compute the gradients of the resulting variables.**<br><br>

In [11]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

Let's calculate the gradient.

In [12]:
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

The function f is piecewise linear in its input a. In other words, for any a, there exists a constant scalar k such that f(a) = k*a, where the value of k depends on the input a; therefore, the gradient can be verified using d/a.

In [13]:
a.grad == d / a

tensor(True)

## **Summary**
* **Deep learning frameworks can automatically compute derivatives: First, we attach gradients to the variables for which we want to compute partial derivatives, then record the calculation of the target value, execute its backpropagation function, and retrieve the resulting gradients.**